### 队列（queue）
新数据项的添加总是发生在一端（通常称为“尾rear”端），而现存数据的移除总是发生在另一端（通常称为首“front”端）

In [ ]:
from fontTools.varLib.mutator import curr
from sympy import true
from torchgen.packaged.autograd.gen_trace_type import tie_return_values


class Queue:
    # 以index 0作为首端，index -1作为尾端
    def __init__(self):
        self.items = []

    def is_empty(self):
        return self.items == []

    def dequeue(self):
        self.items.pop(0)

    def enqueue(self, item):
        self.items.append(item)

    def size(self):
        return len(self.items)

### 队列的应用：“热土豆问题”

In [ ]:
def hot_potato(namelist, num):
    simqueue = []
    for name in namelist:
        simqueue.append(name)

    while len(simqueue) > 1:
        for i in range(num):
            simqueue.append(simqueue.pop(0))

        simqueue.pop(0)

    return simqueue.pop(0)

hot_potato(["Bill", "David", "Susan", "Jane", "Kent", "Brad"], 4)

### 队列的应用：“打印任务”
#### 具体要求为：
- 一个实验室任意一个小时内大约有10个学生在场，每个人发起2次左右的打印，每次打印1~20页
- 打印机的性能是：以草稿模式打印的话，每分钟10页；以正常模式打印的话，每分钟5页
- 打印速度由用户自定义

#### 建模：时间按照秒来流逝
- 按照概率生成打印任务，加入打印队列：每秒钟有$\frac{1}{180}$的概率产生打印任务且每一次打印任务的页数等概率从1~20之间抽取
- 如果打印机空闲，且任务队列不空，则取出队首任务打印，记录该任务的等待时间
- 如果打印机在工作中，则按照打印速度进行1秒打印
- 如果当前任务完成，则打印机进入空闲

In [ ]:
import random
class Printer:
    def __init__(self, ppm):
        # ppm为每分钟打印的页数
        self.page_rate = ppm
        self.current_task = None
        self.time_remaining = 0

    def tick(self):
        if self.current_task is not None:
            self.time_remaining =self.time_remaining - 1
            if self.time_remaining <= 0:
                self.current_task = None

    def busy(self):
        if self.current_task is not None:
            return True
        else: return False

    def start_next(self, new_task):
        self.current_task = new_task
        self.time_remaining = new_task.get_pages() * 60 / self.page_rate

class Task:
    def __init__(self, time):
        self.time_stamp = time
        self.pages = random.randrange(1, 21)

    def get_stamp(self):
        return self.time_stamp

    def get_pages(self):
        return self.pages

    def wait_time(self, current_time):
        return current_time - self.time_stamp

def new_print_task():
    num = random.randrange(1, 181)
    if num == 180: # 其实1~180之间的任意数都可以，代表概率为1/180
        return True
    else:
        return False

def simulation(num_seconds, pages_per_minute):
    lab_printer = Printer(pages_per_minute)
    print_queue = []
    waiting_times = []

    for current_second in range(num_seconds):
        if new_print_task():
            task = Task(current_second)
            print_queue.append(task)

        if (not lab_printer.busy()) and (not print_queue == []):
            next_task = print_queue.pop(0)
            waiting_times.append(next_task.wait_time(current_second))
            lab_printer.start_next(next_task)

        lab_printer.tick()

    average_wait = sum(waiting_times) / len(waiting_times)
    print("Average Wait %6.2f seconds %3d tasks remaining." %(average_wait, len(print_queue)))

for i in range(10):
    simulation(3600, 2)